In [178]:
import pandas as pd
import numpy as np
from datetime import datetime

In [180]:
# Fetch data from web and convert it into pandas data frame
url = 'https://www.forex.com.pk/open_market_rates.asp'
data = pd.read_html(url)
    # currency_data = data[11]

In [182]:
text_date = data[9][1][0]
import re
date = re.search(r"\b\d{1,2}/\d{1,2}/\d{4}\b", text_date).group()


In [186]:
currency = data[11]

In [188]:
currency = currency.rename(columns= {0: "Currecny", 1: "Buying", 2: "Selling"})

In [190]:
currency["Refresh_Date"] = datetime.today().strftime("%d-%m-%Y")
currency["Update_Date"] = pd.to_datetime(date, dayfirst=True).strftime("%d-%m-%Y")

In [192]:
currency

,Currecny,Buying,Selling,Refresh_Date,Update_Date
0,Australian Dollar,194.8900,198.7100,01-02-2026,01-02-2026
1,Bahrain Dinar,743.5000,753.0500,01-02-2026,01-02-2026
2,Canadian Dollar,206.0500,209.6100,01-02-2026,01-02-2026
3,China Yuan,38.1500,40.2500,01-02-2026,01-02-2026
4,Danish Krone,43.3200,43.7200,01-02-2026,01-02-2026
5,Euro,333.8300,338.2600,01-02-2026,01-02-2026
6,Hong Kong Dollar,35.4700,36.3500,01-02-2026,01-02-2026
7,Indian Rupee,2.8200,3.3300,01-02-2026,01-02-2026
8,Japanese Yen,1.8083,1.9095,01-02-2026,01-02-2026
9,Kuwaiti Dinar,907.5500,917.0000,01-02-2026,01-02-2026


In [211]:
import requests
import time
import random
from bs4 import BeautifulSoup

In [213]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

fuel_url = "https://psopk.com/en/fuels/fuel-prices"

In [215]:
def fetch(url, delay=(1,2)):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    time.sleep(random.uniform(*delay))
    return resp.text

In [ ]:
def parse_product_list(html):
    soup = BeautifulSoup(html, "html.parser")
    products = []
    
    for a in soup.select("a[title]"):
        title = a.get("title")
        href = a.get("href")
        if title and href and "/product/" in href  or "daraz" in href:
            products.append({"title": title.scipt(), "url": href})
        
        if not products:
            for img in soup.select("img[alt]"):
                alt = img.get("alt").strip()
                src = img.get("src") or img.get("data-src")
                if alt:
                    products.append({"title": alt, "image": src})

        if not products:
            for tag in soup.select("h2, h3"):
                text = tag.get_text(strip=True)
                if len(text) > 3:
                    products.append({"title": text})

        return products